# LPM-модель: t-тест

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_params
from scipy.stats import t # t-распределение

## t-тест: Значимость коэффициентов

Тестируем гипотезу 
$$
H_0:\beta=0
$$

Тестовая статистика 
$$
t=\frac{\hat{\beta}}{s.e.(\beta)}
$$

Критическое значение
$$
t_{cr}=t_{df=n-k-1}(\alpha)
$$

Гипотеза отвергается если $|t|>t_{cr}$ или $P<\alpha$

### approve equation 1

Для датасета `loanapp` рассморим регрессию **approve на mortno, unem, dep, male, married, yjob, self**

In [ ]:
# подключим датасет loanapp по ссылке 
loanapp_df = pd.read_csv('../../datasets/loanapp.csv', na_values=(' ', '', '  '))
loanapp_df.shape

In [ ]:
#зададим спецификацию модели через формулу
mod_lpm = smf.ols(formula='approve ~ 1 + mortno + unem + dep + male + married + yjob + self', data=loanapp_df)

In [ ]:
# подгонка модели с поправкой на гетероскедастичность
res_lpm_hc = mod_lpm.fit(cov_type='HC3')
print(res_lpm_hc.summary(slim=True))

### Тестируем гипотезу на уровне значимости 1\% (т.е. $\alpha = 0.01$)

## робастные t-статистики для каждого коэффициента

In [ ]:
# робастные t-статистики для каждого коэффициента с округлением до 3-х десятичных знаков
res_lpm_hc.tvalues.round(3)

In [ ]:
# Число наблюдений в модели, число регрессоров и степени свободы для t_cr
res_lpm_hc.nobs, res_lpm_hc.df_model, res_lpm_hc.df_resid

In [ ]:
# Результаты t-теста для коэффициентов (робастные s.e.)
summary_params(res_lpm_hc, alpha=0.01)

In [ ]:
# 1%-критическое значение t-распределения
t_cr = np.round(t.isf(q=0.01/2, df=res_lpm_hc.df_resid), 3)
t_cr

In [ ]:
# проверим значимость коэффициентов используя P-value
df_hc = np.round(summary_params(res_lpm_hc, alpha=0.01), 3)
df_hc['significance'] = df_hc.apply(lambda x: 'Значим' if x['P>|t|']<0.01 else 'Незначим', axis=1)
df_hc

In [ ]:
# проверим значимость коэффициентов используя t_cr
df_hc = np.round(summary_params(res_lpm_hc, alpha=0.01), 3)
df_hc['significance'] = df_hc.apply(lambda x: 'Значим' if np.abs(x['t'])>t_cr else 'Незначим', axis=1)
df_hc

**ВЫВОД**: На уровне значимости 1% значим коэффициент `mortno`

## неробастные t-статистики для каждого коэффициента

In [ ]:
# подгонка модели
res_lpm_ols = mod_lpm.fit(cov_type='nonrobust')

In [ ]:
# Результаты t-теста для коэффициентов (неробастные s.e.)
summary_params(res_lpm_ols, alpha=0.01)

In [ ]:
# проверим значимость коэффициентов используя P-value
df_ols = np.round(summary_params(res_lpm_ols, alpha=0.01), 3)
df_ols['significance'] = df_ols.apply(lambda x: 'Значим' if x['P>|t|']<0.01 else 'Незначим', axis=1)
df_ols

In [ ]:
# проверим значимость коэффициентов используя t_cr
df_ols = np.round(summary_params(res_lpm_ols, alpha=0.01), 3)
df_ols['significance'] = df_ols.apply(lambda x: 'Значим' if np.abs(x['t'])>t_cr else 'Незначим', axis=1)
df_ols

**ВЫВОД**: На уровне значимости 1% значимы коэффициенты: `mortno` и `married`

### Значимость выбранных коэффициентов

Тестируем значимость $\beta_{mortno}$ и $\beta_{male}$

In [ ]:
# робастный t-тест
res_lpm_hc.t_test('mortno=0, male=0')